# IMSTE on a selected single-cell dataset

Choose a source dataset, set the IMSTE parameters, fit directly on its selected high-dimensional `X` matrix, and compare a plain scatter with a KDE density-parent tree plot.


In [1]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = next(
    (parent / "notebooks" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "notebooks" / "imste_single_cell.py").is_file()),
    Path.cwd(),
)
if not (NOTEBOOK_DIR / "imste_single_cell.py").is_file():
    NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import imste_single_cell

CLASP_SRC = next(
    (parent / "src" for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "src" / "clasp").is_dir()),
    None,
)
if CLASP_SRC is not None and str(CLASP_SRC) not in sys.path:
    sys.path.insert(0, str(CLASP_SRC))
from clasp import LabelNeighborGraph

IMSTE_VISUALIZATION_PATH = Path(
    "/Users/f.costa/Code/IteratedMinimumSpanningTree/imste/visualization.py"
)
if not IMSTE_VISUALIZATION_PATH.is_file():
    raise FileNotFoundError(f"IMSTE visualization utilities not found: {IMSTE_VISUALIZATION_PATH}")
_visualization_spec = importlib.util.spec_from_file_location(
    "imste_visualization_tools", IMSTE_VISUALIZATION_PATH
)
imste_visualization = importlib.util.module_from_spec(_visualization_spec)
_visualization_spec.loader.exec_module(imste_visualization)
plot_embedding = imste_single_cell.plot_embedding


In [2]:
AVAILABLE_DATASETS = imste_single_cell.available_datasets()
DATASET_OPTIONS = [(path.name, str(path.resolve())) for path in AVAILABLE_DATASETS]
DEFAULT_DATASET_NAME = "human_pancreas_norm_complexBatch.h5ad"
DEFAULT_DATASET_PATH = next(
    value for name, value in DATASET_OPTIONS if name == DEFAULT_DATASET_NAME
)

import ipywidgets as widgets
DATASET_SELECTOR = widgets.Dropdown(
    options=DATASET_OPTIONS,
    value=DEFAULT_DATASET_PATH,
    description="Dataset:",
    layout=widgets.Layout(width="80%"),
)
display(DATASET_SELECTOR)


Dropdown(description='Dataset:', index=9, layout=Layout(width='80%'), options=(('Immune_ALL_hum_mou.h5ad', '/U…

In [99]:
# Load only the chosen rows from the original high-dimensional X matrix.
MAX_CELLS = 24_000
RANDOM_STATE = 42
MATRIX_KEY = "X"
DATASET_PATH = Path(DATASET_SELECTOR.value).resolve()

X, obs, row_indices = imste_single_cell.load_dataset(
    path=DATASET_PATH,
    matrix_key=MATRIX_KEY,
    obs_keys=tuple(imste_single_cell.available_obs_columns(DATASET_PATH)),
    max_cells=MAX_CELLS,
    random_state=RANDOM_STATE,
)
LABEL_KEY = imste_single_cell.default_label_key(obs)
RANDOMIZE_LABEL_COLORS = True
LABEL_COLOR_SEED = 42
import pandas as pd
label_categories = pd.Categorical(obs[LABEL_KEY]).categories
label_cmap = plt.get_cmap("tab20", max(len(label_categories), 1))
color_order = np.arange(len(label_categories))
if RANDOMIZE_LABEL_COLORS:
    color_order = np.random.default_rng(LABEL_COLOR_SEED).permutation(color_order)
LABEL_COLORS = {
    str(label): label_cmap(color_order[index])
    for index, label in enumerate(label_categories)
}
if obs[LABEL_KEY].isna().any():
    LABEL_COLORS["Missing"] = (0.5, 0.5, 0.5, 1.0)
print(f"Dataset: {DATASET_PATH.name}")
print(f"Representation: {MATRIX_KEY}; shape: {X.shape}; dtype: {X.dtype}")
print(f"Rows loaded: {len(row_indices):,}")
print(f"Plot label: {LABEL_KEY} ({obs[LABEL_KEY].nunique()} classes)")


Dataset: Immune_ALL_hum_mou.h5ad
Representation: X; shape: (24000, 8135); dtype: float32
Rows loaded: 24,000
Plot label: final_annotation (19 classes)


## IMSTE parameters

Edit the values below, then rerun this cell and the fit and plot cells. All estimator parameters are listed explicitly.


In [94]:
n = 128
EMBEDDER_PARAMS = {
    "n_msts": n,
    "n_components": 2,
    "n_epochs": 1000,
    "batch_size": 64_768,
    "learning_rate": 0.05,
    "negative_ratio": 5,
    "lambda_rep": 0.5,
    "random_state": RANDOM_STATE,
    "epsilon": 1e-4,
    "device": "mps",
    "logistic_margin": 1.0,
    "logistic_temperature": 0.5,
    "mst_method": "famst",
    "mst_neighbors": n*2,
    "mst_inter_component_edges": 5,
    "mst_max_neighbors": None,
    "mst_representatives_per_component": 10,
    "progress_callback": None,
    "mst_progress_callback": None,
}
EMBEDDER_PARAMS


{'n_msts': 128,
 'n_components': 2,
 'n_epochs': 1000,
 'batch_size': 64768,
 'learning_rate': 0.05,
 'negative_ratio': 5,
 'lambda_rep': 0.5,
 'random_state': 42,
 'epsilon': 0.0001,
 'device': 'mps',
 'logistic_margin': 1.0,
 'logistic_temperature': 0.5,
 'mst_method': 'famst',
 'mst_neighbors': 256,
 'mst_inter_component_edges': 5,
 'mst_max_neighbors': None,
 'mst_representatives_per_component': 10,
 'progress_callback': None,
 'mst_progress_callback': None}

In [95]:
%%time
estimator, embedding = imste_single_cell.fit_embedding(X, EMBEDDER_PARAMS)
print(f"Embedding shape: {embedding.shape}")
print(f"IMSTE device: {estimator.device_}")

IMSTE source: imste._estimator
MST trees: 128 / 128 requested; method: famst; fallback: none
Device: mps
Embedding shape: (24000, 2)
Graph construction: 17.77 s
Optimization: 49.09 s
Total fit: 67.38 s
Trustworthiness (10 neighbors): 0.8406
Embedding shape: (24000, 2)
IMSTE device: mps
CPU times: user 3min 49s, sys: 32.8 s, total: 4min 22s
Wall time: 1min 43s


In [ ]:
import numpy as np

coordinates = np.asarray(embedding)
span = coordinates.max(axis=0) - coordinates.min(axis=0)
padding = np.maximum(span * 0.04, 1e-6)

fig, axes = plt.subplots(1, 2, figsize=(20, 8), sharex=True, sharey=True)
plot_embedding(
    embedding,
    obs,
    color_keys=(LABEL_KEY,),
    point_size=8,
    point_alpha=0.7,
    ax=axes[0],
    legend_position="bottom",
    label_colors=LABEL_COLORS,
)
imste_single_cell.plot_density_parent_tree(
    embedding,
    obs,
    label_key=LABEL_KEY,
    point_size=8,
    point_alpha=0.0,
    edge_alpha=0.5,
    cross_label_edge_alpha=0.1,
    edge_alpha_exponent=1.2,
    edge_length_std_threshold=0.85,
    k=40,
    ax=axes[1],
    legend_position="bottom",
    label_colors=LABEL_COLORS,
)
for ax in axes:
    ax.set_xlim(coordinates[:, 0].min() - padding[0], coordinates[:, 0].max() + padding[0])
    ax.set_ylim(coordinates[:, 1].min() - padding[1], coordinates[:, 1].max() + padding[1])
fig.subplots_adjust(bottom=0.18, wspace=0.12)
plt.show()


In [ ]:
%%time
from umap import UMAP

UMAP_PARAMS = {
    "n_components": 2,
    "n_neighbors": 128,
    "min_dist": 0.1,
    "metric": "euclidean",
    "random_state": None,
    "low_memory": True,
    "n_jobs": -1,
}
umap_embedding = UMAP(**UMAP_PARAMS).fit_transform(X)
umap_coordinates = np.asarray(umap_embedding)
umap_span = umap_coordinates.max(axis=0) - umap_coordinates.min(axis=0)
umap_padding = np.maximum(umap_span * 0.04, 1e-6)

fig, axes = plt.subplots(1, 2, figsize=(20, 8), sharex=True, sharey=True)
plot_embedding(
    umap_embedding,
    obs,
    color_keys=(LABEL_KEY,),
    point_size=8,
    point_alpha=0.7,
    ax=axes[0],
    legend_position="bottom",
    label_colors=LABEL_COLORS,
)
axes[0].set_title(f"UMAP colored by {LABEL_KEY}")
imste_single_cell.plot_density_parent_tree(
    umap_embedding,
    obs,
    label_key=LABEL_KEY,
    point_size=8,
    point_alpha=0.0,
    edge_alpha=0.5,
    cross_label_edge_alpha=0.1,
    edge_alpha_exponent=1.2,
    edge_length_std_threshold=0.85,
    k=40,
    ax=axes[1],
    legend_position="bottom",
    label_colors=LABEL_COLORS,
)
axes[1].set_xlabel("UMAP 1")
axes[1].set_ylabel("UMAP 2")
for ax in axes:
    ax.set_xlim(
        umap_coordinates[:, 0].min() - umap_padding[0],
        umap_coordinates[:, 0].max() + umap_padding[0],
    )
    ax.set_ylim(
        umap_coordinates[:, 1].min() - umap_padding[1],
        umap_coordinates[:, 1].max() + umap_padding[1],
    )
fig.subplots_adjust(bottom=0.18, wspace=0.12)
plt.show()


In [ ]:
K_LABEL_NEIGHBORS = 30
SHOW_SELF_LINKS = False
SHOW_NODE_COUNTS = False
NODE_ALPHA = 0.6
MIN_NODE_SIZE = 500
MAX_NODE_SIZE = 1200
MIN_EDGE_FRACTION = 0.15

label_positions_path = NOTEBOOK_DIR / f"{DATASET_PATH.stem}_label_positions.json"
node_position_selector = widgets.Dropdown(
    options=[("UMAP", "umap"), ("IMSTE", "imste")],
    value="umap",
    description="Node positions:",
)
display(node_position_selector)

def plot_label_transition_graphs(node_position_source):
    embeddings = {"umap": umap_embedding, "imste": embedding}
    position_embedding = embeddings[node_position_source]
    umap_label_graph = LabelNeighborGraph(
        k=K_LABEL_NEIGHBORS,
        min_edge_fraction=MIN_EDGE_FRACTION,
        min_node_size=MIN_NODE_SIZE,
        max_node_size=MAX_NODE_SIZE,
    ).fit(embeddings["umap"], obs[LABEL_KEY])
    imste_label_graph = LabelNeighborGraph(
        k=K_LABEL_NEIGHBORS,
        min_edge_fraction=MIN_EDGE_FRACTION,
        min_node_size=MIN_NODE_SIZE,
        max_node_size=MAX_NODE_SIZE,
    ).fit(embeddings["imste"], obs[LABEL_KEY])

    label_colors = LABEL_COLORS

    # Position both graph panels using label centroids from the selected embedding.
    umap_label_graph.set_positions_from_embedding(position_embedding)
    umap_label_graph.export_positions(label_positions_path)
    umap_label_graph.import_positions(label_positions_path)
    imste_label_graph.import_positions(label_positions_path)

    fig, axes = plt.subplots(1, 2, figsize=(18, 9))
    umap_label_graph.plot(
        ax=axes[0],
        show_self_links=SHOW_SELF_LINKS,
        show_node_counts=SHOW_NODE_COUNTS,
        node_alpha=NODE_ALPHA,
        label_colors=label_colors,
        title=f"UMAP label transitions (k={K_LABEL_NEIGHBORS})",
    )
    imste_label_graph.plot(
        ax=axes[1],
        show_self_links=SHOW_SELF_LINKS,
        show_node_counts=SHOW_NODE_COUNTS,
        node_alpha=NODE_ALPHA,
        label_colors=label_colors,
        title=f"IMSTE label transitions (k={K_LABEL_NEIGHBORS})",
    )
    fig.suptitle(f"Shared node positions from {node_position_source.upper()}")
    fig.text(
        0.5,
        0.025,
        "Node area is proportional to label count; arrow thickness is the mean fraction of k-NN cells from source to destination label.",
        ha="center",
    )
    fig.subplots_adjust(bottom=0.09, wspace=0.18, top=0.9)
    plt.show()

node_position_output = widgets.interactive_output(
    plot_label_transition_graphs,
    {"node_position_source": node_position_selector},
)
display(node_position_output)
